# 05 — Avaliação: equidade entre as classes

Notebook equivalente a `src/eval/`. Responde à pergunta de pesquisa: **a imagem preserva a assinatura de cada
classe, com equidade?** Número principal: **F1 da pior classe** (decisão do usuário, 2026-10-10), sempre ao lado
do F1 de cada classe e do painel de `project_conventions.md`.

**Branch `omicsfootprint` (R10):** só o necessário para o R10 (`regras_de_testes.md`, "R10: decisões fechadas"). Os
blocos A a F ficam na main. Aqui ficam o painel a partir da matriz de confusão, os 2.000 sorteios do bootstrap pareado
e a BBC-par "melhor imagem − melhor tabular", com as famílias da main lidas de `results/` (só leitura).

Entrada: `data/processed/ofp/coorte_cv.csv` (do `03_images` desta branch) e as previsões fora do fold da main em
`results/`. Saída: só em `results/ofp/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score, matthews_corrcoef

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
# branch omicsfootprint: grava só em results/ofp/; lê da main (results/) as previsões que entram na comparação final
PROCESSED, MAIN, RESULTS = RAIZ / "data" / "processed" / "ofp", RAIZ / "results", RAIZ / "results" / "ofp"
SEED, K, B = 42, 5, 2000  # B = reamostragens do bootstrap
CLASSES = ["CIN", "MSI", "GS", "EBV"]  # = classe_idx 0, 1, 2, 3
PROBS = [f"p_{c}" for c in CLASSES]

coorte = pd.read_csv(PROCESSED / "coorte_cv.csv")
y, fold = coorte.classe_idx.to_numpy(), coorte.fold.to_numpy()

# probabilidades fora do fold dos 4 modelos da main, na ordem do coorte_cv.csv: uma matriz (336, 4) por modelo
P = {}
for m, d in pd.read_csv(MAIN / "predicoes_oof.csv").groupby("modelo", sort=False):
    d = d.set_index("amostra").loc[coorte.amostra]
    assert (d.classe_idx.to_numpy() == y).all() and (d.fold.to_numpy() == fold).all(), m
    P[m] = d[PROBS].to_numpy()
print(f"{len(coorte)} pacientes {dict(zip(CLASSES, np.bincount(y)))} · modelos da main {list(P)}")

336 pacientes {'CIN': np.int64(202), 'MSI': np.int64(59), 'GS': np.int64(46), 'EBV': np.int64(29)} · modelos da main ['cnn_hilbert', 'cnn_vetor', 'logistica', 'xgboost']


## Métricas a partir da matriz de confusão

O bootstrap recalcula o painel 2.000 vezes para cada preditor. Para ficar rápido, as métricas saem direto da
matriz de confusão 4 × 4, com todas as reamostragens de uma vez. O autoteste confere contra o scikit-learn.

In [2]:
def confusoes(y_, yp, idx=None):
    # matrizes de confusão (linha = real, coluna = previsto); com idx (B, n), uma por reamostragem, sem laço
    idx = np.arange(len(y_))[None] if idx is None else idx
    cod = y_[idx] * 4 + yp[idx] + 16 * np.arange(len(idx))[:, None]
    return np.bincount(cod.ravel(), minlength=16 * len(idx)).reshape(-1, 4, 4)


def painel(cm):
    # cm (..., 4, 4) → métricas com forma (...); MCC multiclasse de Gorodkin, o mesmo do scikit-learn
    vp = np.diagonal(cm, axis1=-2, axis2=-1)
    real, prev = cm.sum(-1), cm.sum(-2)
    s, c = cm.sum((-2, -1)), vp.sum(-1)
    f1 = 2 * vp / np.maximum(real + prev, 1)
    mcc = (c * s - (prev * real).sum(-1)) / np.sqrt((s**2 - (prev**2).sum(-1)) * (s**2 - (real**2).sum(-1)))
    return {"MCC": mcc, "acurácia balanceada": (vp / real).mean(-1), "F1 macro": f1.mean(-1), "pior F1": f1.min(-1),
            **{f"F1 {k}": f1[..., i] for i, k in enumerate(CLASSES)}}


# autoteste contra o scikit-learn, numa matriz só e em várias reamostragens de uma vez
rng = np.random.default_rng(SEED)
for _ in range(20):
    yt, yp = rng.integers(0, 4, 60), rng.integers(0, 4, 60)
    m = painel(confusoes(yt, yp)[0])
    assert np.isclose(m["MCC"], matthews_corrcoef(yt, yp))
    assert np.isclose(m["acurácia balanceada"], balanced_accuracy_score(yt, yp))
    assert np.allclose([m[f"F1 {k}"] for k in CLASSES], f1_score(yt, yp, average=None, labels=range(4)))
    idx = rng.integers(0, 60, (5, 60))
    assert np.allclose(painel(confusoes(yt, yp, idx))["MCC"], [matthews_corrcoef(yt[i], yp[i]) for i in idx])

## Os 2.000 sorteios do bootstrap pareado

Os mesmos 2.000 sorteios estratificados da main (mesma semente): cada sorteio tem 336 pacientes com reposição e mantém o
n de cada classe (202 · 59 · 46 · 29). Pareado: os mesmos pacientes sorteados valem para todos os preditores, e a
diferença entre dois preditores é calculada dentro de cada sorteio.

In [3]:
rng = np.random.default_rng(SEED)  # os mesmos 2.000 sorteios da main
idx = np.concatenate([rng.choice(np.flatnonzero(y == c), (B, (y == c).sum())) for c in range(4)], axis=1)
assert (np.bincount(y[idx[0]], minlength=4) == np.bincount(y)).all()

## BBC-CV pareada "melhor imagem − melhor tabular"

A mesma BBC-par do bloco E da main (P0; TSAMARDINOS et al., 2018; seção C.1.5 da PARTE C). Em cada um dos 2.000 sorteios:
1. nos pacientes sorteados, calcula o pior F1 de cada configuração (na imagem, a média das 5 sementes; na logística, a
   execução única) e escolhe a melhor de **cada** família;
2. nos ~124 pacientes que ficaram fora do sorteio (~17 GS), mede a diferença imagem − tabular das duas escolhidas.

A média das 2.000 diferenças é a BBC-par, e os percentis 2,5 e 97,5 dão o intervalo. As previsões vêm da main
(`results/`, só leitura), com as famílias do bloco F: imagem = as 5 configurações com convolução da otimização, o R1a e
o R1b; tabular = a logística, o `largo`, as logísticas do P3a, do P4 e do P3b e as logísticas com a entrada do R1a e do
R1b. A célula reproduz o valor do bloco F gravado na main (autoteste). O R9 e o R10 entram na BBC-par (g), na seção do
R10, abaixo; o R10-p é controle e não entra.

Critério (C.3.2): BBC-par ≤ 0, a imagem não superou a melhor tabular; > 0, a imagem é candidata; com o intervalo inteiro
acima de 0, superior.

In [4]:
SEMENTES = [SEED, 142, 242, 342, 442]


def ler_oof(pasta, nome):
    # previsões fora do fold (336, 4), na ordem do coorte_cv.csv
    proba = np.empty((len(y), len(CLASSES)))
    for k in range(K):
        te = fold == k
        proba[te] = pd.read_csv(pasta / f"{nome}_fold{k}.csv").set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    return proba


# previsões da main (results/, só leitura); a semente 42 da Small CNN (Hilbert) está no predicoes_oof.csv
OTIM = {"profundo": [P["cnn_hilbert"]] + [ler_oof(MAIN / "oof_b", f"cnn_hilbert_s{s}") for s in SEMENTES[1:]],
        **{v: [ler_oof(MAIN / "oof_otim", f"{arq}_s{s}") for s in SEMENTES] for v, arq in [
            ("largo", "largo"), ("largo_profundo", "largo_profundo"), ("profundo + adamw", "profundo_adamw"),
            ("profundo + adamw_focal", "profundo_adamw_focal"), ("profundo + adamw_dro", "profundo_adamw_dro")]}}
ref_o = pd.read_csv(MAIN / "04_models_otimizacao.csv", index_col=0)["pior F1"]
for v, ps in OTIM.items():  # autoteste: reproduz a média do pior F1 de results/04_models_otimizacao.csv
    assert np.isclose(np.mean([painel(confusoes(y, p.argmax(axis=1))[0])["pior F1"] for p in ps]), ref_o[v]), v

VEZES = np.stack([np.bincount(i, minlength=len(y)) for i in idx])  # (2.000, 336): vezes que cada paciente saiu no sorteio


def dentro_fora(yp):
    # pior F1 nos pacientes sorteados (com repetição) e nos que ficaram de fora, em cada um dos 2.000 sorteios
    celula = np.eye(16, dtype=int)[y * 4 + yp]
    return (painel((VEZES @ celula).reshape(-1, 4, 4))["pior F1"],
            painel(((VEZES == 0) @ celula).reshape(-1, 4, 4))["pior F1"])


def familia(preds):
    # preds: {configuração: [classe prevista de cada semente]} → pior F1 fora do sorteio da configuração escolhida dentro
    df = np.array([np.mean([dentro_fora(yp) for yp in yps], axis=0) for yps in preds.values()])  # (config, 2, 2.000)
    escolha = df[:, 0].argmax(axis=0)
    return df[escolha, 1, np.arange(len(idx))], escolha


def bbc_par(imagem, tabular):
    (vi, ei), (vt, et) = familia(imagem), familia(tabular)
    v = vi - vt
    media = {n: np.mean([painel(confusoes(y, yp)[0])["pior F1"] for yp in yps]) for n, yps in (imagem | tabular).items()}
    mais = lambda fam, e: f"{list(fam)[np.bincount(e).argmax()]} ({100 * np.bincount(e).max() / len(idx):.0f}%)"
    return {"diferença sem correção": max(media[n] for n in imagem) - max(media[n] for n in tabular),
            "BBC-par": v.mean(), "IC 2,5%": np.percentile(v, 2.5), "IC 97,5%": np.percentile(v, 97.5),
            "% dos sorteios > 0": 100 * (v > 0).mean(), "imagem: mais escolhida": mais(imagem, ei),
            "tabular: mais escolhida": mais(tabular, et)}


yp_t = P["logistica"].argmax(axis=1)  # autotestes
assert np.array_equal((VEZES[:50] @ np.eye(16, dtype=int)[y * 4 + yp_t]).reshape(-1, 4, 4), confusoes(y, yp_t, idx[:50]))
fora0 = np.setdiff1d(np.arange(len(y)), idx[0])  # o "fora" do 1º sorteio são os pacientes que não saíram nele
assert np.isclose(dentro_fora(yp_t)[1][0], painel(confusoes(y[fora0], yp_t[fora0])[0])["pior F1"])
assert bbc_par({"a": [yp_t]}, {"b": [yp_t]})["BBC-par"] == 0  # a mesma configuração dos dois lados: diferença 0

IMAGEM = {v: [p.argmax(axis=1) for p in ps] for v, ps in OTIM.items() if v != "largo"}
IMAGEM |= {n: [ler_oof(MAIN / "oof_r", f"{n}_s{s}").argmax(axis=1) for s in SEMENTES] for n in ["R1a", "R1b"]}
TABULAR = {"logística": [yp_t], "largo": [p.argmax(axis=1) for p in OTIM["largo"]],
           **{n: [ler_oof(MAIN / pasta, arq).argmax(axis=1)] for n, pasta, arq in [
               ("logística + 24 por cromossomo (P3a)", "oof_p", "P3a_logistica"),
               ("logística em 26 estatísticas globais (P4a)", "oof_p", "P4a_logistica"),
               ("logística + 26 estatísticas globais (P4b)", "oof_p", "P4b_logistica"),
               ("logística, entrada do R1a", "oof_r", "R1a_logistica"),
               ("logística, entrada do R1b", "oof_r", "R1b_logistica"),
               ("logística + frações por braço (P3b)", "oof_p", "P3b_logistica")]}}
assert len(IMAGEM) == 7 and len(TABULAR) == 8
res = pd.DataFrame({"5 com convolução + R1a + R1b × tabular (bloco F da main)": bbc_par(IMAGEM, TABULAR)}).T
ref_f = pd.read_csv(MAIN / "05_eval_bloco_f.csv", index_col=0)  # autoteste: reproduz a leitura (e) do bloco F da main
assert np.isclose(res["BBC-par"].iloc[0], ref_f.loc["(e) 5 com convolução + R1a + R1b × tabular", "BBC-par"])
print("Critério (C.3.2): BBC-par ≤ 0 → a imagem não superou a melhor tabular; > 0 → candidata; intervalo todo > 0 → superior.")
res

Critério (C.3.2): BBC-par ≤ 0 → a imagem não superou a melhor tabular; > 0 → candidata; intervalo todo > 0 → superior.


,diferença sem correção,BBC-par,"IC 2,5%","IC 97,5%",% dos sorteios > 0,imagem: mais escolhida,tabular: mais escolhida
5 com convolução + R1a + R1b × tabular (bloco F da main),-0.039834,-0.031419,-0.117645,0.040552,20.1,profundo + adamw_dro (90%),logística + 26 estatísticas globais (P4b) (50%)


# R10 — avaliação (pré-registro de 2026-11-02)

Diário, "[R10] Pré-registro do R10, do R10-p e da logística sem o X". As previsões do R10, do R10-p e da logística sem o X
vêm de `results/ofp/r10/` (do `04_models` desta branch); as da main, de `results/` (só leitura).
- **Painel:** pior F1 (média e desvio das 5 sementes), F1 de cada classe, MCC e CIN → GS, para a rede base, o R10, o R10-p,
  a logística e a logística sem o X.
- **Diferenças no pior F1, com IC 95% pelo bootstrap pareado** (os 2.000 sorteios acima; média das sementes):
  Perm(R10) = R10 − R10-p; R10 − rede base; Dif(R10) = R10 − logística sem o X; logística sem o X − logística.
- **BBC-par (g):** família imagem = as 5 com convolução + R1a + R1b + R9 + R10 (item 7 da C.4); tabular = a do bloco F + a
  logística sem o X. O R10-p é controle e nunca é escolhido. Autoteste: sem o R10 e sem a logística sem o X, a conta
  reproduz o bloco G da main.
- **Desfecho pelo pré-registro (C.3.5):** confirma o nível 3 (linha (6)) se a BBC-par > 0, a imagem mais escolhida é o
  R10 e Perm(R10) > 0 com IC sem o 0. Com a BBC-par ≤ 0, nível 1: linha (2) se Perm(R10) > 0 com IC sem o 0 (o arranjo
  importa dentro da CNN, mas a imagem fica abaixo da tabela), linha (1) se não. Com a BBC-par > 0 por outra configuração,
  a leitura é pela C.3.5 com o Perm da escolhida. Se o R10 e o R10-p ficarem os dois abaixo da rede base, a leitura
  proposta é que a codificação circular perde informação para esta rede (barras sobrepostas), e não ausência de sinal.

Saída: `results/ofp/05_eval_r10.csv`.

In [5]:
R10 = RESULTS / "r10"  # previsões do 04_models desta branch
PR = {"R10": [ler_oof(R10, f"R10_s{s}") for s in SEMENTES], "R10-p": [ler_oof(R10, f"R10-p_s{s}") for s in SEMENTES]}
P_LOG_X = ler_oof(R10, f"logistica_sem_x_s{SEED}")
R9 = [ler_oof(MAIN / "oof_r", f"R9_s{s}").argmax(axis=1) for s in SEMENTES]  # main, bloco G
assert all(np.allclose(p.sum(axis=1), 1, atol=1e-4) for p in [P_LOG_X, *PR["R10"], *PR["R10-p"]])
CIN, GS = CLASSES.index("CIN"), CLASSES.index("GS")


def resumo(ps):
    # painel médio das sementes (uma execução na logística), com o desvio do pior F1 e os CIN previstos como GS
    m = pd.DataFrame([{k: float(v) for k, v in painel(confusoes(y, p.argmax(axis=1))[0]).items()} for p in ps])
    return {"pior F1": m["pior F1"].mean(), "dp": m["pior F1"].std(), **{f"F1 {c}": m[f"F1 {c}"].mean() for c in CLASSES},
            "MCC": m.MCC.mean(), "CIN → GS": np.mean([confusoes(y, p.argmax(axis=1))[0][CIN, GS] for p in ps])}


def dif_media(pares, met="pior F1"):
    # média, sobre as sementes, da diferença a − b; IC 95% pelo bootstrap pareado (como no bloco D da main)
    pont = np.mean([painel(confusoes(y, a)[0])[met] - painel(confusoes(y, b)[0])[met] for a, b in pares])
    dist = np.mean([painel(confusoes(y, a, idx))[met] - painel(confusoes(y, b, idx))[met] for a, b in pares], axis=0)
    return {"diferença": pont, "IC 2,5%": np.percentile(dist, 2.5), "IC 97,5%": np.percentile(dist, 97.5)}


BASE = "rede base (profundo + adamw_dro)"
tab = pd.DataFrame({n: resumo(ps) for n, ps in {BASE: OTIM["profundo + adamw_dro"], **PR, "logística": [P["logistica"]],
                                                 "logística sem o X": [P_LOG_X]}.items()}).T
display(tab.round(3))

am = lambda ps: [p.argmax(axis=1) for p in ps]  # noqa: E731
r10, r10p, base = am(PR["R10"]), am(PR["R10-p"]), am(OTIM["profundo + adamw_dro"])
lx, lg = P_LOG_X.argmax(axis=1), P["logistica"].argmax(axis=1)
difs = pd.DataFrame({"Perm(R10) = R10 − R10-p": dif_media(list(zip(r10, r10p))),
                     "R10 − rede base": dif_media(list(zip(r10, base))),
                     "Dif(R10) = R10 − logística sem o X": dif_media([(a, lx) for a in r10]),
                     "logística sem o X − logística": dif_media([(lx, lg)])}).T
display(difs.round(3))

IMG_G, TAB_G = IMAGEM | {"R9": R9}, TABULAR | {"logística sem o X": [lx]}
ref_g = pd.read_csv(MAIN / "05_eval_bloco_g.csv", index_col=0)  # autoteste: sem o R10, reproduz o bloco G da main
assert np.isclose(bbc_par(IMG_G, TABULAR)["BBC-par"], ref_g.loc["(f) 5 com convolução + R1a + R1b + R9 × tabular", "BBC-par"])
bbc = bbc_par(IMG_G | {"R10": r10}, TAB_G)

perm = difs.loc["Perm(R10) = R10 − R10-p"]
perm_pos, escolhida = perm["IC 2,5%"] > 0, bbc["imagem: mais escolhida"].rsplit(" (", 1)[0]
if bbc["BBC-par"] <= 0:
    desfecho = ("nível 1, linha (2): o arranjo importa dentro da CNN, mas a imagem não superou a melhor tabular" if perm_pos
                else "nível 1, linha (1): a imagem não superou a melhor tabular, e o arranjo do R10 não carrega informação")
elif escolhida == "R10":
    desfecho = ("nível 3, linha (6): CONFIRMA" if perm_pos
                else "BBC-par > 0 pelo R10, mas Perm(R10) com IC no 0: vale a rede na imagem circular, não o arranjo (leitura da linha (5))")
else:
    desfecho = f"BBC-par > 0 pela escolhida {escolhida}, não pelo R10: ler pela C.3.5 com o Perm da escolhida"
print(f"Desfecho pelo pré-registro (C.3.5): {desfecho}. Perm(R10) {perm['diferença']:+.3f} "
      f"({perm['IC 2,5%']:+.3f} a {perm['IC 97,5%']:+.3f}); BBC-par {bbc['BBC-par']:+.3f} ({bbc['IC 2,5%']:+.3f} a "
      f"{bbc['IC 97,5%']:+.3f}); imagem mais escolhida: {bbc['imagem: mais escolhida']}")
if max(tab.loc["R10", "pior F1"], tab.loc["R10-p", "pior F1"]) < tab.loc[BASE, "pior F1"]:
    print("O R10 e o R10-p ficaram os dois abaixo da rede base: leitura proposta, a codificação circular perde informação.")
RESULTS.mkdir(parents=True, exist_ok=True)
pd.concat({"painel": tab, "diferenças no pior F1": difs,
           "BBC-par (g)": pd.DataFrame({"5 com convolução + R1a + R1b + R9 + R10 × tabular + logística sem o X": bbc}).T}
          ).to_csv(RESULTS / "05_eval_r10.csv")
pd.Series(bbc, name="BBC-par (g)").to_frame()

,pior F1,dp,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,CIN → GS
rede base (profundo + adamw_dro),0.717,0.015,0.926,0.962,0.717,0.922,0.834,17.6
R10,0.622,0.037,0.892,0.965,0.622,0.812,0.762,24.2
R10-p,0.599,0.019,0.885,0.971,0.599,0.817,0.755,28.2
logística,0.750,NaN,0.924,0.956,0.750,0.951,0.842,18.0
logística sem o X,0.757,NaN,0.927,0.956,0.757,0.951,0.847,17.0


,diferença,"IC 2,5%","IC 97,5%"
Perm(R10) = R10 − R10-p,0.023,-0.020,0.066
R10 − rede base,-0.096,-0.147,-0.042
Dif(R10) = R10 − logística sem o X,-0.135,-0.203,-0.063
logística sem o X − logística,0.007,0.000,0.024


Desfecho pelo pré-registro (C.3.5): nível 1, linha (1): a imagem não superou a melhor tabular, e o arranjo do R10 não carrega informação. Perm(R10) +0.023 (-0.020 a +0.066); BBC-par -0.039 (-0.120 a +0.036); imagem mais escolhida: profundo + adamw_dro (73%)
O R10 e o R10-p ficaram os dois abaixo da rede base: leitura proposta, a codificação circular perde informação.


,BBC-par (g)
diferença sem correção,-0.039834
BBC-par,-0.039407
"IC 2,5%",-0.120141
"IC 97,5%",0.0364
% dos sorteios > 0,15.3
imagem: mais escolhida,profundo + adamw_dro (73%)
tabular: mais escolhida,logística + 26 estatísticas globais (P4b) (50%)
